In [1]:
#kept this cell as it has all the calcualtions 

#animations and examples are not visable as inorder to be able to upload to github
#it needs to be smaller than a certain file size. had to clear the output
#run all the cells inorder to see animations and experiments.
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# Geometry: distance to segment
# -------------------------------
def point_to_segment_distance(x, a, b):
    """
    Distance from point x to line segment [a, b]
    """
    v = b - a
    w = x - a
    vv = np.dot(v, v)

    if vv == 0.0:
        return np.linalg.norm(x - a)

    t = np.dot(w, v) / vv
    t = np.clip(t, 0.0, 1.0)
    q = a + t * v
    return np.linalg.norm(x - q)

# -------------------------------
# Wall penalty (quadratic band)
# -------------------------------
def wall_penalty(x, a, b, R):
    d = point_to_segment_distance(x, a, b)
    if d <= R:
        return 0.5 * (R - d)**2
    else:
        return 0.0


# -------------------------------
# Wall penalty (Student Made)
# -------------------------------
def wall_penalty_log(x, a, b, R):
    d = point_to_segment_distance(x, a, b)
    if d <= R:

        e = 0.01
        return np.log(R / d + e)
    else:
        return 0.0
        

def wall_penalty_exp(x, a, b, R):
    d = point_to_segment_distance(x, a, b)
    
    if d <= R:

        s = 0.5
        return np.exp(- (d ** 2) / (2 * s ** 2))
    else:
        return 0.0




# -------------------------------
# Grid setup
# -------------------------------
xmin, xmax = -5, 5
ymin, ymax = -5, 5
N = 100

xs = np.linspace(xmin, xmax, N)
ys = np.linspace(ymin, ymax, N)
X, Y = np.meshgrid(xs, ys)

# Wall definition (horizontal wall)
a = np.array([-3.0, 0.0])
b = np.array([ 3.0, 0.0])
R = 1.0  # influence radius

# -------------------------------
# Evaluate cost field
#added the new fnctions that way we can plot all 3 functions for a more
#direct comparsion
# -------------------------------
C = np.zeros_like(X)

#da log and exponitional functions
C_log = np.zeros_like(X)
C_exp = np.zeros_like(X)

for i in range(N):
    for j in range(N):
        x = np.array([X[i, j], Y[i, j]])
        C[i, j] = wall_penalty(x, a, b, R)
        C_log[i, j] = wall_penalty_log(x, a, b, R)
        C_exp[i, j] = wall_penalty_exp(x, a, b, R)

# -------------------------------
# Numerical gradient
# -------------------------------
dCy, dCx = np.gradient(C, ys, xs)

dCy_log, dCx_log = np.gradient(C_log, ys, xs)
dCy_exp, dCx_exp = np.gradient(C_exp, ys, xs)







## Floor plans to use

Plot floor-plan

In [ ]:
def plot_floor_plan(walls, start=None, goal=None):
    plt.figure(figsize=(6, 6))
    for a, b in walls:
        a = np.array(a)
        b = np.array(b)
        plt.plot([a[0], b[0]], [a[1], b[1]], "k-", linewidth=2)

    if start is not None:
        plt.plot(start[0], start[1], "go", markersize=8, label="Start")

    if goal is not None:
        plt.plot(goal[0], goal[1], "ro", markersize=8, label="Goal")

    plt.axis("equal")
    plt.grid(True)
    plt.legend()
    plt.title("Assignment Floor Plan")
    plt.show()


### Selectable Floor Plan Layouts

In [ ]:
# -----------------------------------------------------------------------------
# Floor plan options for the assignment
# -----------------------------------------------------------------------------

import numpy as np

def load_floor_plan(name="baseline"):
    """
    Load a predefined floor plan.

    Available options:
      - "baseline"  : corridor + corner (recommended default)
      - "zigzag"    : narrow zig-zag corridor (oscillation test)
      - "deadend"   : dead-end with escape (local minimum trap)
      - "symmetric" : symmetric obstacle trap (pathological case)

    Returns:
      walls : list of (a, b) line segments
      start : np.array shape (2,)
      goal  : np.array shape (2,)
    """

    if name == "baseline":
        walls = [
            # Outer boundary
            ([-5, -5], [ 5, -5]),
            ([ 5, -5], [ 5,  5]),
            ([ 5,  5], [-5,  5]),
            ([-5,  5], [-5, -5]),

            # Interior corridor
            ([-1, -5], [-1,  2]),
            ([-1,  2], [ 3,  2]),
            ([ 3,  2], [ 3, -2]),
        ]
        start = [-4.0, -4.0]
        goal  = [ 4.0,  4.0]

    elif name == "zigzag":
        walls = [
            # Outer boundary
            ([-6, -4], [ 6, -4]),
            ([ 6, -4], [ 6,  4]),
            ([ 6,  4], [-6,  4]),
            ([-6,  4], [-6, -4]),

            # Zig-zag corridor
            ([-4, -3], [-1, -3]),
            ([-1, -3], [-1,  0]),
            ([-1,  0], [ 2,  0]),
            ([ 2,  0], [ 2,  3]),
            ([ 2,  3], [ 5,  3]),
        ]
        start = [-5.0, -3.5]
        goal  = [ 5.0,  3.5]

    elif name == "deadend":
        walls = [
            # Outer boundary
            ([-6, -6], [ 6, -6]),
            ([ 6, -6], [ 6,  6]),
            ([ 6,  6], [-6,  6]),
            ([-6,  6], [-6, -6]),

            # Dead-end chamber
            ([-2, -6], [-2,  2]),
            ([-2,  2], [ 2,  2]),
            ([ 2,  2], [ 2, -2]),
            ([ 2, -2], [-4, -2]),
            ([-4, -2], [-4,  4]),
        ]
        start = [ 0.0, -5.0]
        goal  = [ 5.0,  5.0]

    elif name == "symmetric":
        walls = [
            # Outer boundary
            ([-6, -6], [ 6, -6]),
            ([ 6, -6], [ 6,  6]),
            ([ 6,  6], [-6,  6]),
            ([-6,  6], [-6, -6]),

            # Symmetric obstacles
            ([-2, -2], [-2,  2]),
            ([ 2, -2], [ 2,  2]),
            ([-2,  2], [ 2,  2]),
        ]
        start = [ 0.0, -5.0]
        goal  = [ 0.0,  5.0]

    else:
        raise ValueError(f"Unknown floor plan: {name}")

    # Convert to NumPy arrays
    walls = [(np.array(a, dtype=float), np.array(b, dtype=float)) for a, b in walls]
    start = np.array(start, dtype=float)
    goal  = np.array(goal, dtype=float)

    return walls, start, goal


In [ ]:
#@title Floor Plan Selection

floor_plan = "zigzag" #@param ["baseline", "zigzag", "deadend", "symmetric"]


walls, start, goal = load_floor_plan(floor_plan)

plot_floor_plan(walls, start, goal)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np



#bassically calculates the distance from the x to the goal the higher it is the worse it is
def goal_cost(x, goal):
    #the distance formula basically, halfing to make it not run at mach 10 towards the goal might change it
    return  np.sum((x - goal)**2)


# -------------------------------
# Wall penalty (quadratic band)
# -------------------------------
def wall_penalty(x, a, b, R):
    d = point_to_segment_distance(x, a, b)
    if d <= R:
        return 0.5 * (R - d)**2
    else:
        return 0.0


# -------------------------------
# Wall penalty (Student Made)
# -------------------------------
def wall_penalty_log(x, a, b, R):
    d = point_to_segment_distance(x, a, b)
    if d <= R:

        e = 0.01
        return np.log(R / (d + e))
    else:
        return 0.0
        

#4th
#starting s =0.5
#increasing s to 2
#made it more fuzzy, increasing not the way to go

#4rth, exp 2nd
#s = 0.3
#this one seems the best balance bewteen range and desnity

#it was a bug
#i forgot to change from the quad to exp in the code
#thats why it was almost unseeable

def wall_penalty_exp(x, a, b, R):
    d = point_to_segment_distance(x, a, b)
    
    if d <= R:

        s = 0.3
        return np.exp(- (d ** 2) / (2 * s ** 2))
    else:
        return 0.0



#x is the current pos, walls is walls, penelaty is the selected type, weights is how strong the repulsion is
# r is the max distance before it has no wieght
def wall_cost(x,walls,penalty,weight,r):
    total = 0.0

    #goes through all the walls
    for a, b in walls:

        if penalty == "quad":
            total += wall_penalty(x, a, b, r)
        elif penalty == "log":
            total += wall_penalty_log(x, a, b, r)
        elif penalty == "exp":
            total += wall_penalty_exp(x, a, b, r)

    return total * weight



def total_cost(x,goal,walls,penalty,weight,r):
    c_goal = goal_cost(x,goal)

    c_walls = wall_cost(x,walls,penalty,weight,r)

    return c_goal + c_walls




xmin, xmax = -6, 6
ymin, ymax = -6, 6
N = 50

xs = np.linspace(xmin, xmax, N)
ys = np.linspace(ymin, ymax, N)
X, Y = np.meshgrid(xs, ys)










In [ ]:
#used AI to get my prevouis notebook code and condense it
#then used it to modify it to allow multie particle support
#



# Load environment
walls, start, goal = load_floor_plan("zigzag")

# TASK 1: Multi-Particle Setup
N = 5  # Number of particles

# Initialize particles near the original start position
np.random.seed(42)
positions = np.random.uniform(
    [start[0] - 0.3, start[1] - 0.3],
    [start[0] + 0.3, start[1] + 0.3],
    size=(N, 2)
)


# Simulation parameters
learning_rate = 0.001
max_iterations = 2500
tolerance = 0.1

# Define penalty configurations
penalty_configs = [
    {"penalty": "log", "weight": 80.0, "r": 0.2},
    {"penalty": "quad", "weight": 400.0, "r": 0.5},
    {"penalty": "exp", "weight": 100.0, "r": 2}
]


def run_multi_particle_simulation(initial_positions, goal, walls, penalty, weight, r, 
                                   learning_rate, max_iterations, tolerance):
    """Run gradient descent simulation for MULTIPLE particles"""
    N = initial_positions.shape[0]
    positions = initial_positions.copy()
    trajectory = [positions.copy()]
    
    for iteration in range(max_iterations):
        new_positions = np.zeros_like(positions)
        
        for particle_i in range(N):
            current_pos = positions[particle_i]
            
            # Check if reached goal
            distance_to_goal = np.linalg.norm(current_pos - goal)
            if distance_to_goal < tolerance:
                new_positions[particle_i] = current_pos
                continue
            
            # Compute gradient
            epsilon = 1e-5
            grad = np.zeros(2)
            
            for dim in range(2):
                x_plus = current_pos.copy()
                x_minus = current_pos.copy()
                x_plus[dim] += epsilon
                x_minus[dim] -= epsilon
                
                cost_plus = total_cost(x_plus, goal, walls, penalty, weight, r)
                cost_minus = total_cost(x_minus, goal, walls, penalty, weight, r)
                
                grad[dim] = (cost_plus - cost_minus) / (2 * epsilon)
            
            new_positions[particle_i] = current_pos - learning_rate * grad
        
        positions = new_positions
        trajectory.append(positions.copy())
    
    return np.array(trajectory)


def plot_multi_particle_trajectory(trajectory, walls, goal, penalty_name):
    """Plot trajectories for multiple particles"""
    n_steps, N, _ = trajectory.shape
    
    plt.figure(figsize=(10, 10))
    
    # Plot walls
    for a, b in walls:
        plt.plot([a[0], b[0]], [a[1], b[1]], 'k-', linewidth=2)
    
    colors = plt.cm.rainbow(np.linspace(0, 1, N))
    
    for particle_i in range(N):
        particle_traj = trajectory[:, particle_i, :]
        
        plt.plot(particle_traj[:, 0], particle_traj[:, 1], '-', 
                color=colors[particle_i], alpha=0.6, linewidth=2,
                label=f'Particle {particle_i}')
        
        plt.scatter(particle_traj[0, 0], particle_traj[0, 1],
                   color=colors[particle_i], s=150, marker='o',
                   edgecolors='black', linewidth=2, zorder=5)
        
        plt.scatter(particle_traj[-1, 0], particle_traj[-1, 1],
                   color=colors[particle_i], s=150, marker='s',
                   edgecolors='black', linewidth=2, zorder=5)
    
    plt.scatter(goal[0], goal[1], c='red', s=300, marker='*', 
               edgecolors='black', linewidth=2, label='Goal', zorder=6)
    
    plt.xlabel('x', fontsize=12)
    plt.ylabel('y', fontsize=12)
    plt.title(f'Multi-Particle Gradient Descent - {penalty_name} penalty (N={N})', fontsize=14)
    plt.axis('equal')
    plt.legend(loc='best', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.show()


# Run simulations
for config in penalty_configs:

    
    trajectory = run_multi_particle_simulation(
        initial_positions=positions,
        goal=goal,
        walls=walls,
        penalty=config['penalty'],
        weight=config['weight'],
        r=config['r'],
        learning_rate=learning_rate,
        max_iterations=max_iterations,
        tolerance=tolerance
    )
    
    plot_multi_particle_trajectory(trajectory, walls, goal, config['penalty'])

    #So far this only adds support for multiple particples
    # they dont interact yet but are instead randomly seperated around the start location to add a bit of smovment.
    #they are all unqiue color coded for easy viewing
    #intrestinlgy both exp and quad seem to have a consitant apporch towards the goal as the colored lines meld together unlike log
    # which seems to be more chaotic and depednent on the start



In [ ]:
#the social weights that are gonna be used



#meant to be soft kinda like normal quadratic
def social_cost_quadratic(xi, xj, R=1.0, k_social=5.0):

    dij = np.linalg.norm(xi - xj)
    
    if dij <= R:
        return 0.5 * k_social * (R - dij)**2
    else:
        return 0.0


# expontial is like regular wall expontila in which it has longer range
def social_cost_exponential(xi, xj, A=2.0, B=0.5):

    dij = np.linalg.norm(xi - xj)
    return A * np.exp(-dij / B)


    

#inverse seems to act very similar to log in the way that it is primarly short range
def social_cost_inverse(xi, xj, A=1.0, p=2, epsilon=0.1):

    dij = np.linalg.norm(xi - xj)
    return A / (dij + epsilon)**p

#calculates the graident of all the particples earby the particle that is called upon

def gradient_social_numerical(xi, xj, social_cost_func, epsilon=1e-5, **params):

    grad = np.zeros(2)
    
    for dim in range(2):
        xi_plus = xi.copy()
        xi_minus = xi.copy()
        xi_plus[dim] += epsilon
        xi_minus[dim] -= epsilon
        
        cost_plus = social_cost_func(xi_plus, xj, **params)
        cost_minus = social_cost_func(xi_minus, xj, **params)
        
        grad[dim] = (cost_plus - cost_minus) / (2 * epsilon)
    
    return grad


#modification of the orginal simulation using socail wieghts this time
def run_multi_particle_simulation_with_social(initial_positions, goal, walls, 
                                               penalty, weight, r,
                                               social_cost_func, social_params,
                                               learning_rate, max_iterations, tolerance):
    N = initial_positions.shape[0]
    positions = initial_positions.copy()
    trajectory = [positions.copy()]
    
    for iteration in range(max_iterations):
        new_positions = np.zeros_like(positions)
        
        for particle_i in range(N):
            current_pos = positions[particle_i]
            
            # Check if reached goal
            distance_to_goal = np.linalg.norm(current_pos - goal)
            if distance_to_goal < tolerance:
                new_positions[particle_i] = current_pos
                continue
            
            # Compute gradient from GOAL
            epsilon = 1e-5
            grad = np.zeros(2)
            
            for dim in range(2):
                x_plus = current_pos.copy()
                x_minus = current_pos.copy()
                x_plus[dim] += epsilon
                x_minus[dim] -= epsilon
                
                cost_plus = total_cost(x_plus, goal, walls, penalty, weight, r)
                cost_minus = total_cost(x_minus, goal, walls, penalty, weight, r)
                
                grad[dim] = (cost_plus - cost_minus) / (2 * epsilon)
            
            # ADD: Compute gradient from SOCIAL FORCES
            for particle_j in range(N):
                if particle_j != particle_i:  # Don't interact with self
                    grad_social = gradient_social_numerical(
                        current_pos, 
                        positions[particle_j],
                        social_cost_func,
                        **social_params
                    )
                    grad += grad_social  # Add social force gradient
            
            # Update position
            new_positions[particle_i] = current_pos - learning_rate * grad
        
        positions = new_positions
        trajectory.append(positions.copy())
    
    return np.array(trajectory)

social_force_configs = [
    {
        "name": "Quadratic Repulsion",
        "func": social_cost_quadratic,
        "params": {"R": 1.5, "k_social": 5.0}
    },
    {
        "name": "Exponential",
        "func": social_cost_exponential,
        "params": {"A": 3.0, "B": 0.5}
    },
    {
        "name": "Inverse Distance",
        "func": social_cost_inverse,
        "params": {"A": 0.6, "p": 1, "epsilon": 0.1}
    }
]


def plot_multi_particle_trajectory(trajectory, walls, goal, penalty_name, social_name=None):
    """Plot trajectories for multiple particles"""
    n_steps, N, _ = trajectory.shape
    
    plt.figure(figsize=(10, 10))
    
    # Plot walls
    for a, b in walls:
        plt.plot([a[0], b[0]], [a[1], b[1]], 'k-', linewidth=2)
    
    colors = plt.cm.rainbow(np.linspace(0, 1, N))
    
    for particle_i in range(N):
        particle_traj = trajectory[:, particle_i, :]
        
        plt.plot(particle_traj[:, 0], particle_traj[:, 1], '-', 
                color=colors[particle_i], alpha=0.6, linewidth=2,
                label=f'Particle {particle_i}')
        
        plt.scatter(particle_traj[0, 0], particle_traj[0, 1],
                   color=colors[particle_i], s=150, marker='o',
                   edgecolors='black', linewidth=2, zorder=5)
        
        plt.scatter(particle_traj[-1, 0], particle_traj[-1, 1],
                   color=colors[particle_i], s=150, marker='s',
                   edgecolors='black', linewidth=2, zorder=5)
    
    plt.scatter(goal[0], goal[1], c='red', s=300, marker='*', 
               edgecolors='black', linewidth=2, label='Goal', zorder=6)
    
    if social_name:
        title = f'Multi-Particle: {penalty_name} penalty + {social_name} social force (N={N})'
    else:
        title = f'Multi-Particle: {penalty_name} penalty (N={N})'
    
    plt.xlabel('x', fontsize=12)
    plt.ylabel('y', fontsize=12)
    plt.title(title, fontsize=12)
    plt.axis('equal')
    plt.legend(loc='best', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.show()




#o is log, 1 is quad, 2 exp
#very convient for r
test_penalty = penalty_configs[1]  

for social_config in social_force_configs:
    
    trajectory = run_multi_particle_simulation_with_social(
        initial_positions=positions,
        goal=goal,
        walls=walls,
        penalty=test_penalty['penalty'],
        weight=test_penalty['weight'],
        r=test_penalty['r'],
        social_cost_func=social_config['func'],
        social_params=social_config['params'],
        learning_rate=learning_rate,
        max_iterations=max_iterations,
        tolerance=tolerance
    )
    
    print(f"Trajectory shape: {trajectory.shape}")
    plot_multi_particle_trajectory(
        trajectory, walls, goal, 
        test_penalty['penalty'],
        social_config['name']
    )


    #after reviewing the animations we can see that invese is the strongstet of the forcs where thier spheres of influence
    # barely overlap
    #compared to quadractic replusion in which they sometimes overlap but overall keep their distance
    #expontial is by far the weakest with the particles staying in each others spheres of influence much longer than 
    #than the other repulsions



In [ ]:
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

mpl.rcParams['animation.embed_limit'] = 50  # MB


#just loops over the prevouis cell animating that one so we can see how it ends up
for social_config in social_force_configs:

    trajectory = run_multi_particle_simulation_with_social(
        initial_positions=positions,
        goal=goal,
        walls=walls,
        penalty=test_penalty['penalty'],
        weight=test_penalty['weight'],
        r=test_penalty['r'],
        social_cost_func=social_config['func'],
        social_params=social_config['params'],
        learning_rate=learning_rate,
        max_iterations=max_iterations,
        tolerance=tolerance
    )

    n_steps, N, _ = trajectory.shape
    colors = plt.cm.rainbow(np.linspace(0, 1, N))
    trail_length = 10
    congestion_threshold = 2.0

    fig, ax = plt.subplots(figsize=(6, 6))

    for a, b in walls:
        ax.plot([a[0], b[0]], [a[1], b[1]], 'k-', linewidth=3)

    ax.scatter(goal[0], goal[1], c='red', s=300, marker='*',
               edgecolors='black', linewidth=2, zorder=6, label='Goal')

    padding = 1.0
    ax.set_xlim(trajectory[:,:,0].min() - padding, trajectory[:,:,0].max() + padding)
    ax.set_ylim(trajectory[:,:,1].min() - padding, trajectory[:,:,1].max() + padding)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(f'Multi-Particle: Quadratic penalty + {social_config["name"]} social force', fontsize=11)
    ax.set_xlabel('x')
    ax.set_ylabel('y')

    trail_lines   = [ax.plot([], [], '-', color=colors[i], alpha=0.4, linewidth=1.5)[0] for i in range(N)]
    particle_dots = [ax.plot([], [], 'o', color=colors[i], markersize=10,
                             markeredgecolor='black', markeredgewidth=1.5, zorder=5)[0] for i in range(N)]
    congestion_circles = [plt.Circle((0,0), 0.0, color='red', alpha=0.0, zorder=4) for _ in range(N)]
    for c in congestion_circles:
        ax.add_patch(c)

    step_text = ax.text(0.02, 0.96, '', transform=ax.transAxes, fontsize=10,
                        verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    def init():
        for line in trail_lines: line.set_data([], [])
        for dot in particle_dots: dot.set_data([], [])
        step_text.set_text('')
        return trail_lines + particle_dots + congestion_circles + [step_text]

    def update(frame):
        start = max(0, frame - trail_length)
        for i in range(N):
            trail_lines[i].set_data(trajectory[start:frame+1, i, 0], trajectory[start:frame+1, i, 1])
            particle_dots[i].set_data([trajectory[frame, i, 0]], [trajectory[frame, i, 1]])
            max_cong = max(
                ((1.0 - np.linalg.norm(trajectory[frame,i] - trajectory[frame,j]) / congestion_threshold)
                 for j in range(N) if i != j and
                 np.linalg.norm(trajectory[frame,i] - trajectory[frame,j]) < congestion_threshold),
                default=0.0)
            congestion_circles[i].center = (trajectory[frame, i, 0], trajectory[frame, i, 1])
            congestion_circles[i].radius = max_cong * 0.8
            congestion_circles[i].set_alpha(max_cong * 0.4)
        step_text.set_text(f'Step: {frame}/{n_steps-1}')
        return trail_lines + particle_dots + congestion_circles + [step_text]

    anim = FuncAnimation(fig, update, frames=range(0, n_steps, 6),
                         init_func=init, interval=100, blit=True)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))





In [ ]:
#now we move onto task1.4
#weve been using istropic forces now we need to dinfe ansiotropic forces

In [ ]:
def social_cost_anisotropic(xi, xj, vi, beta=4.0, A=2.0, B=0.5):
    """Anisotropic social force — upweights particles in direction of motion."""
    dij = np.linalg.norm(xi - xj)
    if dij < 1e-8:
        return 0.0

    # Base isotropic cost (exponential)
    phi = A * np.exp(-dij / B)

    # Directional bias
    v_norm = np.linalg.norm(vi)
    if v_norm < 1e-8:
        return phi  # no velocity → fall back to isotropic

    v_hat = vi / v_norm
    diff_hat = (xj - xi) / dij
    dot = np.dot(v_hat, diff_hat)
    weight = 1.0 + beta * max(0.0, dot)

    return weight * phi


def run_simulation_anisotropic(initial_positions, goal, walls,
                                penalty, weight, r,
                                beta, A, B,
                                learning_rate, max_iterations, tolerance):
    N = initial_positions.shape[0]
    positions = initial_positions.copy()
    # track previous positions to estimate velocity
    prev_positions = initial_positions.copy()
    trajectory = [positions.copy()]

    for iteration in range(max_iterations):
        new_positions = np.zeros_like(positions)

        for i in range(N):
            current_pos = positions[i]

            if np.linalg.norm(current_pos - goal) < tolerance:
                new_positions[i] = current_pos
                continue

            # Velocity estimate from last step
            vi = current_pos - prev_positions[i]

            # Goal + wall gradient
            epsilon = 1e-5
            grad = np.zeros(2)
            for dim in range(2):
                xp = current_pos.copy(); xm = current_pos.copy()
                xp[dim] += epsilon;      xm[dim] -= epsilon
                grad[dim] = (total_cost(xp, goal, walls, penalty, weight, r) -
                             total_cost(xm, goal, walls, penalty, weight, r)) / (2 * epsilon)

            # Anisotropic social gradient
            for j in range(N):
                if j != i:
                    for dim in range(2):
                        xp = current_pos.copy(); xm = current_pos.copy()
                        xp[dim] += epsilon;      xm[dim] -= epsilon
                        cp = social_cost_anisotropic(xp, positions[j], vi, beta, A, B)
                        cm = social_cost_anisotropic(xm, positions[j], vi, beta, A, B)
                        grad[dim] += (cp - cm) / (2 * epsilon)

            new_positions[i] = current_pos - learning_rate * grad

        prev_positions = positions.copy()
        positions = new_positions
        trajectory.append(positions.copy())

    return np.array(trajectory)

In [ ]:


mpl.rcParams['animation.embed_limit'] = 50

# ── Pick your social force config here ──────────────────────────────────────
task4_social_config = social_force_configs[2]  # 0=Quadratic, 1=Exponential, 2=Inverse

# ── Anisotropic social force (uses same base function as isotropic) ──────────
def social_cost_anisotropic(xi, xj, vi, beta=2.0, social_func=None, social_params={}):
    dij = np.linalg.norm(xi - xj)
    if dij < 1e-8:
        return 0.0
    phi = social_func(xi, xj, **social_params)
    v_norm = np.linalg.norm(vi)
    if v_norm < 1e-8:
        return phi
    v_hat = vi / v_norm
    diff_hat = (xj - xi) / dij
    dot = np.dot(v_hat, diff_hat)
    return (1.0 + beta * max(0.0, dot)) * phi

def run_simulation_anisotropic(initial_positions, goal, walls,
                                penalty, weight, r,
                                beta, social_func, social_params,
                                learning_rate, max_iterations, tolerance):
    N = initial_positions.shape[0]
    positions = initial_positions.copy()
    prev_positions = initial_positions.copy()
    trajectory = [positions.copy()]

    for iteration in range(max_iterations):
        new_positions = np.zeros_like(positions)
        for i in range(N):
            current_pos = positions[i]
            if np.linalg.norm(current_pos - goal) < tolerance:
                new_positions[i] = current_pos
                continue
            vi = current_pos - prev_positions[i]
            epsilon = 1e-5
            grad = np.zeros(2)
            for dim in range(2):
                xp = current_pos.copy(); xm = current_pos.copy()
                xp[dim] += epsilon;      xm[dim] -= epsilon
                grad[dim] = (total_cost(xp, goal, walls, penalty, weight, r) -
                             total_cost(xm, goal, walls, penalty, weight, r)) / (2 * epsilon)
            for j in range(N):
                if j != i:
                    for dim in range(2):
                        xp = current_pos.copy(); xm = current_pos.copy()
                        xp[dim] += epsilon;      xm[dim] -= epsilon
                        grad[dim] += (social_cost_anisotropic(xp, positions[j], vi, beta, social_func, social_params) -
                                      social_cost_anisotropic(xm, positions[j], vi, beta, social_func, social_params)) / (2 * epsilon)
            new_positions[i] = current_pos - learning_rate * grad
        prev_positions = positions.copy()
        positions = new_positions
        trajectory.append(positions.copy())

    return np.array(trajectory)

# ── Run both simulations ─────────────────────────────────────────────────────
print(f"Running Isotropic  with {task4_social_config['name']}...")
traj_iso = run_multi_particle_simulation_with_social(
    initial_positions=positions, goal=goal, walls=walls,
    penalty=test_penalty['penalty'], weight=test_penalty['weight'], r=test_penalty['r'],
    social_cost_func=task4_social_config['func'],
    social_params=task4_social_config['params'],
    learning_rate=learning_rate, max_iterations=max_iterations, tolerance=tolerance
)

print(f"Running Anisotropic with {task4_social_config['name']}...")
traj_ani = run_simulation_anisotropic(
    initial_positions=positions, goal=goal, walls=walls,
    penalty=test_penalty['penalty'], weight=test_penalty['weight'], r=test_penalty['r'],
    beta=2.0,
    social_func=task4_social_config['func'],
    social_params=task4_social_config['params'],
    learning_rate=learning_rate, max_iterations=max_iterations, tolerance=tolerance
)

# ── Side-by-side animation ───────────────────────────────────────────────────
n_steps = min(traj_iso.shape[0], traj_ani.shape[0])
N = positions.shape[0]
colors = plt.cm.rainbow(np.linspace(0, 1, N))
trail_length = 10
congestion_threshold = 2.0

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
trajs  = [traj_iso, traj_ani]
titles = ['Isotropic', 'Anisotropic']

all_trails = []; all_dots = []; all_circles = []

for ax, title, traj in zip(axes, titles, trajs):
    for a, b in walls:
        ax.plot([a[0], b[0]], [a[1], b[1]], 'k-', linewidth=3)
    ax.scatter(goal[0], goal[1], c='red', s=200, marker='*', zorder=6)
    padding = 1.0
    ax.set_xlim(traj[:,:,0].min()-padding, traj[:,:,0].max()+padding)
    ax.set_ylim(traj[:,:,1].min()-padding, traj[:,:,1].max()+padding)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_title(f'{title} Social Force\nWall: {test_penalty["penalty"]} | Social: {task4_social_config["name"]}', fontsize=10)

    trails  = [ax.plot([], [], '-',  color=colors[i], alpha=0.4, linewidth=1.5)[0] for i in range(N)]
    dots    = [ax.plot([], [], 'o',  color=colors[i], markersize=9,
                       markeredgecolor='black', markeredgewidth=1.5, zorder=5)[0] for i in range(N)]
    circles = [plt.Circle((0,0), 0.0, color='red', alpha=0.0, zorder=4) for _ in range(N)]
    for c in circles: ax.add_patch(c)

    all_trails.append(trails); all_dots.append(dots); all_circles.append(circles)

step_text = axes[0].text(0.02, 0.96, '', transform=axes[0].transAxes, fontsize=9,
                          verticalalignment='top',
                          bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

def init():
    for trails, dots in zip(all_trails, all_dots):
        for line in trails: line.set_data([], [])
        for dot  in dots:   dot.set_data([], [])
    step_text.set_text('')
    return [item for sub in all_trails+all_dots+all_circles for item in sub] + [step_text]

def update(frame):
    start = max(0, frame - trail_length)
    for idx, traj in enumerate(trajs):
        for i in range(N):
            all_trails[idx][i].set_data(traj[start:frame+1, i, 0], traj[start:frame+1, i, 1])
            all_dots[idx][i].set_data([traj[frame, i, 0]], [traj[frame, i, 1]])
            max_cong = max(
                ((1.0 - np.linalg.norm(traj[frame,i] - traj[frame,j]) / congestion_threshold)
                 for j in range(N) if i != j and
                 np.linalg.norm(traj[frame,i] - traj[frame,j]) < congestion_threshold),
                default=0.0)
            all_circles[idx][i].center = (traj[frame, i, 0], traj[frame, i, 1])
            all_circles[idx][i].radius = max_cong * 0.8
            all_circles[idx][i].set_alpha(max_cong * 0.4)
    step_text.set_text(f'Step: {frame}/{n_steps-1}')
    return [item for sub in all_trails+all_dots+all_circles for item in sub] + [step_text]

anim = FuncAnimation(fig, update, frames=range(0, n_steps, 6),
                     init_func=init, interval=100, blit=True)
plt.tight_layout()
plt.close(fig)
display(HTML(anim.to_jshtml()))

In [ ]:
# Comparative Analysis

# 1. How does directional dependence affect the symmetry of the social force field?
# In isotropic, where directions don't matter, particles end up more bunched together
# and tend to be fairly symmetric in their force, only ever shifting left or right
# due to multiple particles. So in the final spread of the isotropic we can see a
# fairly symmetric distribution. Comparatively, in anisotropic we can see that
# direction influences the symmetry greatly. We see this most notably at the end
# once the particles start taking different directions and during the middle sections
# where they move around chaotically. The symmetry that was present originally is
# broken as particles move in vastly different directions to one another. This is
# because the dot product term weights repulsion based on alignment with the
# velocity direction.

# 2. Does the anisotropic model reduce oscillations or deadlocks? Why?
# Not really. Looking at the simulation both suffer from deadlock and cannot reach
# the goal properly. Though it is notable that in isotropic the particles are more
# bunched up than in anisotropic, at least when examining the quadratic repulsion
# formula. Deadlocks still persist because the shared goal forces all particles to
# converge to the same point regardless of social force model. As for oscillations,
# it does seem to still happen but less so in anisotropic, notably shown by the
# smaller size of the congestion circles.

# 3. How does each model behave in narrow corridors or near bottlenecks?
# Isotropic:
# - Quadratic: particles behave similar to jelly marbles, for the most part never
#   entering the influence of each other and keeping their distance. Near deadlocks
#   and narrow corridors they line up neatly one after another.
# - Inverse: more harsh than quadratic by a large margin, with particles barely
#   getting close to each other. In bottlenecks they almost wait until the first
#   particle is further in before entering the corridor.
# - Exponential: surprisingly similar to inverse until the last part, likely due to
#   the range of influence found in exponential. Once that range hurdle is passed
#   they line up similar to quadratic though much closer. Exponential seems to be
#   between quadratic and inverse in terms of behaviors in tight corridors.
#
# Anisotropic:
# - Quadratic: behaves similar to isotropic quadratic until the bottleneck, where
#   it suddenly starts to act similar to inverse, with no particle entering until
#   the first particle is halfway through.
# - Inverse: a lot of modification was needed for anisotropic inverse to work, as
#   too much force caused particles to phase through the walls. The force is visibly
#   stronger in anisotropic than isotropic. Around the midpoint the purple particle
#   rapidly oscillates before stabilizing, compared to isotropic where only minor
#   oscillation happens.
# - Exponential: this one has the biggest change between isotropic and anisotropic.
#   Exponential in anisotropic results in particles being very far apart, with them
#   very rarely being forced together. Unlike inverse it does not vibrate when nearby,
#   instead flowing naturally back into position. This is also visible in the ending
#   where anisotropic particles are much further apart than in isotropic.

# 4. What trade-offs arise when introducing velocity-dependent interactions?
# A evident trade-off is computation time. Due to isotropic not needing to account
# for velocity it spends less time computing each iteration. While running the
# simulation it was evident that anisotropic took more time to render than isotropic.
# Another trade-off is sensitivity. Isotropic is more resistant to parameter changes
# than anisotropic. Parameter tuning was needed to ensure that particles would not
# fly off violently in anisotropic.